# PULSE — Review Queue

Cross-target review surface:
- Read all review queues (aliases, allocator_types, edges, ontology_terms, signals)
- Preview the diff each override would produce against `_effective` views
- Write reviewer decisions to `human_reviews` (append-only)

**Doctrine**: Normalized rows are never mutated. Overrides are applied at query time via views.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import json
from agents.db import get_conn
from agents.reviews.queue_writer import read_queue, queue_counts, VALID_TARGET_TYPES
from agents.reviews.override_applier import get_review_status, list_latest_reviews, ingest_decisions

con = get_conn()
print('Connected to PULSE DuckDB')

In [ ]:
# Queue counts summary
counts = queue_counts()
print('Review queue sizes:')
for tt, cnt in sorted(counts.items()):
    print(f'  {tt}: {cnt} items')

In [ ]:
# Review decisions in DB
db_status = get_review_status(con)
print('\nHuman review decisions in DB:')
for tt, decisions in sorted(db_status.items()):
    print(f'  {tt}: {decisions}')

In [ ]:
# Aliases queue — most common review type
alias_queue = read_queue('aliases')
print(f'\nAlias queue items: {len(alias_queue)}')

if alias_queue:
    df = pd.DataFrame(alias_queue)
    display(df[['entity_id', 'current_value', 'confidence', 'reason', 'surfaced_at']].head(20))

In [ ]:
# Ontology terms queue
onto_queue = read_queue('ontology_terms')
print(f'\nOntology terms queue: {len(onto_queue)} items')
if onto_queue:
    df_onto = pd.DataFrame(onto_queue)
    display(df_onto[['entity_id', 'current_value', 'confidence', 'reason']].head(20))

In [ ]:
# Preview: what would a 'confirm' decision look like on the top alias?
if alias_queue:
    sample = alias_queue[0]
    print('Sample alias item:')
    print(json.dumps(sample, indent=2))
    
    print('\nTo confirm this alias, create a file decisions.jsonl with:')
    decision_example = {
        'entity_id': sample['entity_id'],
        'target_type': 'alias',
        'decision': 'confirm',
        'reviewer': 'your_name',
        'override_reason': 'Verified match — same institution, different name variant'
    }
    print(json.dumps(decision_example))

In [ ]:
# Write a sample decision directly (for demonstration — replace with your decisions)
# Uncomment and modify to actually write decisions:

# import json, uuid, tempfile, os
# from pathlib import Path
# 
# decisions = [
#     {
#         'entity_id': 'YOUR_ENTITY_ID',
#         'target_type': 'alias',
#         'decision': 'confirm',
#         'reviewer': 'your_name',
#         'override_reason': 'Verified match'
#     }
# ]
# 
# with tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=False, encoding='utf-8') as f:
#     for d in decisions:
#         f.write(json.dumps(d) + '\n')
#     tmp_path = Path(f.name)
# 
# inserted = ingest_decisions(tmp_path, con, reviewer='notebook')
# print(f'Inserted {inserted} review decisions')
# os.unlink(tmp_path)

print('Decision writer ready. Uncomment the block above to write decisions.')

In [ ]:
# Effective view preview — see the impact of current reviews
try:
    eff_alloc = con.execute("""
        SELECT canonical_name, allocator_type, geography, review_decision, reviewer
        FROM allocators_effective
        WHERE review_decision IS NOT NULL
        ORDER BY canonical_name
    """).fetchdf()
    print(f'Allocators with active reviews: {len(eff_alloc)}')
    display(eff_alloc)
except Exception as e:
    print(f'No reviews applied yet: {e}')